# 9. 결정트리와 앙상블

> **제9장** · **이론편 대응: 9.1절(결정트리), 9.5절(앙상블)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: 없음

---

## 이 장에서 하는 일

7~8장에서 선형 모델과 K-Means를 다뤘다. 이번에는 **완전히 다른 발상**의 모델이다.

> **"조건을 물어 나가며 나눈다"**

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 결정트리의 발상 | 9.1절 |
| 2 | **지니 불순도 손계산 검증** ★ | 9.1절 |
| 3 | 분할 기준을 직접 찾기 | 9.1절 |
| 4 | **트리는 왜 과대적합하는가** ★ | 8.5절, 9.1절 |
| 5 | 앙상블 — 여럿을 모으면 | 9.5절 |
| 6 | 배깅과 랜덤 포레스트 | 9.5절 |
| 7 | 부스팅 | 9.5절 |
| 8 | 특성 중요도 | 9.5절 |

**2절과 4절이 핵심이다.** 왜 트리가 학습 데이터를 100% 맞히면서
시험 데이터에서는 무너지는지, 그리고 앙상블이 그것을 어떻게 푸는지 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)

import sklearn
print(f"scikit-learn {sklearn.__version__}")

---

## 1. 결정트리의 발상 — 이론편 9.1절

**스무고개와 같다.** 질문을 하나씩 던져 답의 범위를 좁혀 간다.

```
소득이 5000만원 이상인가?
├─ 예 → 근속연수가 10년 이상인가?
│        ├─ 예 → 승진 (확률 0.85)
│        └─ 아니오 → 미승진 (확률 0.60)
└─ 아니오 → 미승진 (확률 0.78)
```

**선형 모델과 무엇이 다른가**

| | 선형 모델 (7장) | 결정트리 |
|---|---|---|
| 결정 경계 | 직선·평면 | **계단 모양** |
| 특성 스케일 | 표준화 필요 | **불필요** |
| 특성 간 상호작용 | 직접 만들어 줘야 | **자동으로 잡음** |
| 해석 | 계수의 크기 | **규칙 그대로 읽음** |

**스케일링이 필요 없다는 점**이 실무에서 큰 장점이다.
6장에서 다룬 전처리 중 상당 부분을 건너뛸 수 있다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# 선형으로는 나눌 수 없는 데이터
np.random.seed(42)
n = 300
X_xor = np.random.randn(n, 2)
y_xor = ((X_xor[:, 0] > 0) ^ (X_xor[:, 1] > 0)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- 원본 데이터 ---
ax = axes[0]
for label, color, marker in [(0, "#1E40AF", "o"), (1, "#DC2626", "s")]:
    m = y_xor == label
    ax.scatter(X_xor[m, 0], X_xor[m, 1], c=color, marker=marker,
               s=25, alpha=0.7, label=f"클래스 {label}")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_title("데이터 (XOR 형태)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 결정 경계 비교 ---
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, (name, model) in zip(axes[1:], [
        ("로지스틱 회귀 (선형)", LogisticRegression()),
        ("결정트리", DecisionTreeClassifier(max_depth=4, random_state=42))]):
    model.fit(X_xor, y_xor)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    for label, color, marker in [(0, "#1E40AF", "o"), (1, "#DC2626", "s")]:
        m = y_xor == label
        ax.scatter(X_xor[m, 0], X_xor[m, 1], c=color, marker=marker,
                   s=20, alpha=0.7, edgecolors="white", linewidth=0.4)
    acc = model.score(X_xor, y_xor)
    ax.set_title(f"{name}\n정확도 {acc:.3f}")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 70)
print("8장의 XOR 문제를 떠올려 보자")
print("=" * 70)
print("  8장에서는 단일 퍼셉트론이 XOR 을 못 풀어 은닉층을 추가했다.")
print("  결정트리는 **한 층씩 나누는 것만으로** 이 문제를 푼다.")
print()
print("  경계가 계단 모양인 것이 보인다 — 축에 평행한 선으로만 나누기 때문이다.")
print("  이것이 트리의 장점이자 한계다.")

---

## 2. 지니 불순도 ★ — 이론편 9.1절

**어떤 질문이 좋은 질문인가.** 나눈 뒤 각 그룹이 **한쪽으로 쏠릴수록** 좋다.

이 "쏠림"을 재는 척도가 **불순도(impurity)**다.

$$\text{Gini} = 1 - \sum_i p_i^2$$

이론편 9.1절에서 손으로 계산한 값들을 확인한다.

In [ ]:
import numpy as np


def gini(counts):
    """지니 불순도 (이론편 9.1절)

    counts: 각 클래스의 개수, 예 [8, 2]
    """
    n = sum(counts)
    if n == 0:
        return 0.0
    return 1.0 - sum((c / n) ** 2 for c in counts)


def entropy(counts):
    """엔트로피 (이론편 6.4절)"""
    n = sum(counts)
    if n == 0:
        return 0.0
    return -sum((c / n) * np.log2(c / n) for c in counts if c > 0)


print("=" * 70)
print("불순도 계산 — 이론편 9.1절 값 검증")
print("=" * 70)
print(f"{'분포':<20}{'지니':<14}{'엔트로피':<14}{'해석'}")
print("-" * 70)

cases = [
    ([10, 0],  "완전히 한쪽"),
    ([9, 1],   "거의 한쪽"),
    ([8, 2],   "치우침"),
    ([7, 3],   "약간 치우침"),
    ([5, 5],   "완전히 반반"),
]

for counts, desc in cases:
    g, e = gini(counts), entropy(counts)
    print(f"{str(counts):<20}{g:<14.4f}{e:<14.4f}{desc}")

print("-" * 70)
print()

# 이론편 값 검증
assert abs(gini([10, 0]) - 0.0) < 1e-9
assert abs(gini([5, 5]) - 0.5) < 1e-9
assert abs(gini([8, 2]) - 0.32) < 1e-9
assert abs(entropy([5, 5]) - 1.0) < 1e-9
print("[OK] 이론편 9.1절 값과 일치")
print()
print("두 척도의 관계")
print("  둘 다 '순수하면 0, 반반이면 최대'")
print("  지니 최대 = 0.5, 엔트로피 최대 = 1.0 (2클래스 기준)")
print("  지니가 계산이 빨라 기본값으로 쓰인다 (log 연산이 없으므로)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))

p = np.linspace(0.001, 0.999, 200)
gini_vals = 1 - (p**2 + (1-p)**2)
entropy_vals = -(p * np.log2(p) + (1-p) * np.log2(1-p))
error_vals = np.minimum(p, 1-p)

ax.plot(p, gini_vals, linewidth=2.5, color="#1E40AF", label="지니 불순도")
ax.plot(p, entropy_vals / 2, linewidth=2.5, color="#EA580C",
        linestyle="--", label="엔트로피 ÷ 2")
ax.plot(p, error_vals, linewidth=2.5, color="#0D9488",
        linestyle=":", label="오분류율")

ax.axvline(0.5, color="gray", linewidth=1, linestyle="--")
ax.text(0.51, 0.05, "반반 지점", fontsize=8, color="gray")

ax.set_xlabel("한 클래스의 비율 p")
ax.set_ylabel("불순도")
ax.set_title("세 가지 불순도 척도 (2클래스)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("세 곡선 모두 p=0.5 에서 최대, 0과 1에서 0이다.")
print()
print("차이점")
print("  지니·엔트로피는 곡선 — 미세한 개선도 반영한다")
print("  오분류율은 꺾은선 — 다수 클래스가 안 바뀌면 변화가 없다")
print()
print("  → 트리 분할에는 곡선인 지니·엔트로피를 쓴다")

### 정보 이득 — 좋은 분할이란

분할의 좋고 나쁨은 **불순도가 얼마나 줄었는가**로 잰다.

$$\text{이득} = \text{Gini}(\text{부모}) - \sum_k \frac{n_k}{n}\text{Gini}(\text{자식}_k)$$

자식의 불순도를 **크기로 가중평균**한다는 점이 중요하다.
작은 그룹이 순수해도 전체에 미치는 영향은 작기 때문이다.

In [ ]:
import numpy as np


def information_gain(parent, children):
    """정보 이득 계산 (이론편 9.1절)"""
    n = sum(parent)
    weighted = sum(sum(c) / n * gini(c) for c in children)
    return gini(parent) - weighted, weighted


print("=" * 78)
print("분할의 좋고 나쁨 — 부모 [10, 10] 을 나눈다")
print("=" * 78)

parent = [10, 10]
print(f"부모 지니: {gini(parent):.4f}  (완전히 반반이라 최대)")
print()
print(f"{'분할':<16}{'왼쪽':<14}{'오른쪽':<14}{'자식 가중평균':<18}{'이득'}")
print("-" * 78)

splits = [
    ("완벽 분할",  [10, 0], [0, 10]),
    ("좋은 분할",  [8, 2],  [2, 8]),
    ("보통 분할",  [7, 3],  [3, 7]),
    ("나쁜 분할",  [5, 5],  [5, 5]),
    ("치우친 분할", [10, 8], [0, 2]),
]

for name, left, right in splits:
    gain, weighted = information_gain(parent, [left, right])
    print(f"{name:<16}{str(left):<14}{str(right):<14}{weighted:<18.4f}{gain:.4f}")

print("-" * 78)
print()

# 이론편 값 검증
gain_perfect, _ = information_gain([10, 10], [[10, 0], [0, 10]])
gain_good, _ = information_gain([10, 10], [[8, 2], [2, 8]])
gain_bad, _ = information_gain([10, 10], [[5, 5], [5, 5]])

assert abs(gain_perfect - 0.5) < 1e-9
assert abs(gain_good - 0.18) < 1e-9
assert abs(gain_bad - 0.0) < 1e-9
print("[OK] 이론편 9.1절 값과 일치 (0.5 / 0.18 / 0.0)")
print()
print("[읽는 법]")
print("  이득이 클수록 좋은 분할이다.")
print("  '나쁜 분할'은 이득이 0 — 나눠도 불순도가 그대로다.")
print()
print("마지막 '치우친 분할'을 보라.")
print("  오른쪽이 작지만 순수하다. 그래도 이득은 크지 않다.")
print("  가중평균이므로 큰 쪽(왼쪽)의 불순도가 지배하기 때문이다.")

---

## 3. 분할 기준을 직접 찾기 — 이론편 9.1절

트리는 **모든 특성과 모든 분할점을 시도해** 이득이 가장 큰 것을 고른다.
직접 구현해 보자.

In [ ]:
import numpy as np


def best_split(X, y, verbose=False):
    """최선의 분할을 찾는다 (이론편 9.1절)

    모든 특성, 모든 분할점을 시도한다.
    """
    n_samples, n_features = X.shape
    parent_counts = [np.sum(y == c) for c in np.unique(y)]
    parent_gini = gini(parent_counts)

    best = {"gain": -1, "feature": None, "threshold": None}
    trials = []

    for feat in range(n_features):
        # 후보 분할점: 정렬한 값들의 중간점
        values = np.unique(X[:, feat])
        thresholds = (values[:-1] + values[1:]) / 2

        for thr in thresholds:
            left_mask = X[:, feat] <= thr
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            left_counts = [np.sum(y[left_mask] == c) for c in np.unique(y)]
            right_counts = [np.sum(y[right_mask] == c) for c in np.unique(y)]

            n_l, n_r = left_mask.sum(), right_mask.sum()
            weighted = (n_l / n_samples) * gini(left_counts) + \
                       (n_r / n_samples) * gini(right_counts)
            gain = parent_gini - weighted

            trials.append((feat, thr, gain))
            if gain > best["gain"]:
                best = {"gain": gain, "feature": feat, "threshold": thr,
                        "left": left_counts, "right": right_counts}

    if verbose:
        print(f"  시도한 분할 후보: {len(trials)}개")
    return best, trials


# 작은 예제로 확인
np.random.seed(0)
X_small = np.array([
    [3.0, 1.0], [4.0, 1.5], [5.0, 2.0], [6.0, 1.2],
    [7.0, 3.0], [8.0, 3.5], [9.0, 3.2], [10.0, 4.0],
])
y_small = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("=" * 70)
print("최선의 분할 찾기")
print("=" * 70)
print("데이터")
print(f"{'특성0':<10}{'특성1':<10}{'클래스'}")
for xi, yi in zip(X_small, y_small):
    print(f"{xi[0]:<10.1f}{xi[1]:<10.1f}{yi}")
print()

best, trials = best_split(X_small, y_small, verbose=True)

print()
print(f"최선의 분할")
print(f"  특성   : {best['feature']}번")
print(f"  기준값 : {best['threshold']:.2f}")
print(f"  이득   : {best['gain']:.4f}")
print(f"  왼쪽   : {best['left']}   (지니 {gini(best['left']):.4f})")
print(f"  오른쪽 : {best['right']}   (지니 {gini(best['right']):.4f})")
print()
print("이 분할로 완전히 나뉜다 — 양쪽 모두 지니가 0이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 특성별 이득 곡선 ---
ax = axes[0]
for feat, color in [(0, "#1E40AF"), (1, "#EA580C")]:
    pts = [(t, g) for f, t, g in trials if f == feat]
    if pts:
        ts, gs = zip(*sorted(pts))
        ax.plot(ts, gs, marker="o", markersize=4, linewidth=2,
                color=color, label=f"특성 {feat}")

ax.axvline(best["threshold"], color="#DC2626", linestyle="--", linewidth=1.5)
ax.text(best["threshold"] + 0.1, best["gain"] * 0.55,
        f"최선\n{best['threshold']:.1f}", fontsize=8, color="#DC2626")
ax.set_xlabel("분할 기준값")
ax.set_ylabel("정보 이득")
ax.set_title("분할점에 따른 이득")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 오른쪽: 데이터와 분할선 ---
ax = axes[1]
for label, color, marker in [(0, "#1E40AF", "o"), (1, "#DC2626", "s")]:
    m = y_small == label
    ax.scatter(X_small[m, 0], X_small[m, 1], c=color, marker=marker,
               s=90, alpha=0.8, edgecolors="white", linewidth=1.2,
               label=f"클래스 {label}")

if best["feature"] == 0:
    ax.axvline(best["threshold"], color="#0D9488", linewidth=2.5,
               label=f"분할: 특성0 ≤ {best['threshold']:.1f}")
else:
    ax.axhline(best["threshold"], color="#0D9488", linewidth=2.5,
               label=f"분할: 특성1 ≤ {best['threshold']:.1f}")

ax.set_xlabel("특성 0")
ax.set_ylabel("특성 1")
ax.set_title("찾아낸 분할선")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("이 예제는 두 특성 모두로 완벽히 나뉜다.")
print("  실제 데이터는 이렇게 깔끔하지 않으므로 여러 번 나눠야 한다.")
print()
print("[계산량]")
print("  특성 f개, 표본 n개면 후보가 f x (n-1) 개")
print("  깊이가 깊어질수록 이 계산이 반복된다 — 트리 학습이 느린 이유")

---

## 4. 트리는 왜 과대적합하는가 ★ — 이론편 8.5절

**결정트리를 제한 없이 키우면 학습 데이터를 100% 맞힌다.**

당연하다. 계속 나누다 보면 결국 각 잎에 표본이 하나씩만 남기 때문이다.
**그런데 그것이 좋은 모델인가?**

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 노이즈가 있는 데이터 — 실제 상황을 흉내
X, y = make_classification(
    n_samples=800, n_features=12, n_informative=6, n_redundant=2,
    n_classes=2, random_state=42, flip_y=0.08)      # flip_y: 8%는 라벨이 뒤집힘

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print("=" * 78)
print("트리 깊이에 따른 과대적합")
print("=" * 78)
print(f"학습 {len(X_tr)}개 / 시험 {len(X_te)}개, 특성 {X.shape[1]}개")
print("  (라벨의 8%는 일부러 뒤집어 노이즈를 넣었다)")
print()
print(f"{'깊이':<12}{'학습 정확도':<16}{'시험 정확도':<16}{'격차':<12}{'잎 개수'}")
print("-" * 78)

depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
history = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr_acc = accuracy_score(y_tr, tree.predict(X_tr))
    te_acc = accuracy_score(y_te, tree.predict(X_te))
    n_leaves = tree.get_n_leaves()
    history.append((d, tr_acc, te_acc, n_leaves))
    label = str(d) if d else "제한 없음"
    print(f"{label:<12}{tr_acc:<16.4f}{te_acc:<16.4f}{tr_acc-te_acc:<+12.4f}{n_leaves}")

print("-" * 78)
print()

best_idx = max(range(len(history)), key=lambda i: history[i][2])
best_d = history[best_idx][0]
print(f"시험 정확도가 가장 높은 깊이: {best_d}")
print(f"  그때 격차: {history[best_idx][1] - history[best_idx][2]:+.4f}")
print()
print(f"제한 없이 키우면 학습 {history[-1][1]:.4f} / 시험 {history[-1][2]:.4f}")
print(f"  학습 데이터를 완벽히 외웠지만 새 데이터에서는 무너진다.")
print()
print("이것이 이론편 8.5절의 과대적합이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

xs = list(range(len(history)))
labels = [str(h[0]) if h[0] else "∞" for h in history]
tr_accs = [h[1] for h in history]
te_accs = [h[2] for h in history]
leaves = [h[3] for h in history]

# --- 왼쪽: 정확도 곡선 ---
ax = axes[0]
ax.plot(xs, tr_accs, marker="o", linewidth=2.5,
        color="#DC2626", label="학습 정확도")
ax.plot(xs, te_accs, marker="s", linewidth=2.5,
        color="#0D9488", label="시험 정확도")

best_i = int(np.argmax(te_accs))
ax.axvline(best_i, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(best_i + 0.15, 0.72, f"최적\n깊이 {labels[best_i]}",
        fontsize=8, color="#1E40AF")

ax.set_xticks(xs)
ax.set_xticklabels(labels)
ax.set_xlabel("최대 깊이")
ax.set_ylabel("정확도")
ax.set_title("깊어질수록 학습만 좋아진다")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 잎 개수 ---
ax = axes[1]
ax.plot(xs, leaves, marker="o", linewidth=2.5, color="#EA580C")
ax.set_xticks(xs)
ax.set_xticklabels(labels)
ax.set_xlabel("최대 깊이")
ax.set_ylabel("잎(leaf) 개수")
ax.set_yscale("log")
ax.set_title("모델 복잡도")
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("왼쪽 그래프는 이론편 8.5절의 전형적인 과대적합 곡선이다.")
print("  24장 SFT 실습에서 본 것과 같은 모양이다.")
print()
print(f"오른쪽: 잎이 {leaves[0]}개에서 {leaves[-1]}개까지 늘어난다.")
print("  잎이 많다 = 규칙이 잘게 쪼개졌다 = 학습 데이터에 맞춰졌다")

### 트리가 특히 과대적합하기 쉬운 이유

**결정 경계가 자유롭기 때문이다.**

선형 모델은 직선 하나로만 나눌 수 있어 표현력이 제한된다.
트리는 **원하는 만큼 잘게 나눌 수 있어** 어떤 데이터든 외울 수 있다.

**대책 두 가지**

| 방법 | 어떻게 |
|---|---|
| 가지치기(pruning) | `max_depth`, `min_samples_leaf` 등으로 제한 |
| **앙상블** | **여러 트리를 만들어 평균 — 5절부터** |

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("=" * 78)
print("가지치기 매개변수")
print("=" * 78)
print()
print(f"{'매개변수':<24}{'뜻':<32}{'효과'}")
print("-" * 78)
params_desc = [
    ("max_depth", "최대 깊이", "가장 직관적"),
    ("min_samples_split", "분할에 필요한 최소 표본", "작은 그룹은 안 나눔"),
    ("min_samples_leaf", "잎에 있어야 할 최소 표본", "극단적 분할 방지"),
    ("max_leaf_nodes", "최대 잎 개수", "전체 크기 제한"),
    ("ccp_alpha", "비용 복잡도 가지치기", "학습 후 잘라냄"),
]
for a, b, c in params_desc:
    print(f"{a:<24}{b:<32}{c}")
print("-" * 78)
print()

print("실제 효과 비교")
print(f"{'설정':<36}{'학습':<12}{'시험':<12}{'잎'}")
print("-" * 78)

configs = [
    ("제한 없음", {}),
    ("max_depth=4", {"max_depth": 4}),
    ("min_samples_leaf=20", {"min_samples_leaf": 20}),
    ("max_leaf_nodes=16", {"max_leaf_nodes": 16}),
    ("ccp_alpha=0.01", {"ccp_alpha": 0.01}),
    ("깊이4 + 잎최소10", {"max_depth": 4, "min_samples_leaf": 10}),
]

for name, kw in configs:
    t = DecisionTreeClassifier(random_state=42, **kw).fit(X_tr, y_tr)
    tr = accuracy_score(y_tr, t.predict(X_tr))
    te = accuracy_score(y_te, t.predict(X_te))
    print(f"{name:<36}{tr:<12.4f}{te:<12.4f}{t.get_n_leaves()}")

print("-" * 78)
print()
print("제한을 걸면 학습 정확도는 떨어지지만 시험 정확도는 오른다.")
print("  '덜 외우고 더 일반화한다'는 뜻이다.")

---

## 5. 앙상블 — 여럿을 모으면 — 이론편 9.5절

**한 사람의 판단보다 여러 사람의 다수결이 낫다.**

35장의 Self-Consistency와 같은 발상이다. 거기서 계산했던 것을 다시 보자.

개별 정확도가 $p$인 모델 $k$개의 다수결이 맞을 확률은 이항분포로 계산된다.
**$p > 0.5$이면 $k$가 커질수록 정확해진다.**

**단, 조건이 있다.**

In [ ]:
from math import comb
import numpy as np


def majority_accuracy(p, k):
    """개별 정확도 p인 모델 k개의 다수결 정확도

    35장 5절에서 쓴 것과 같은 계산이다.
    """
    threshold = k // 2 + 1
    return sum(comb(k, i) * p**i * (1-p)**(k-i) for i in range(threshold, k+1))


print("=" * 78)
print("다수결의 효과 (이론편 9.5절)")
print("=" * 78)
print()
print(f"{'개별 정확도':<16}{'1개':<12}{'5개':<12}{'11개':<12}{'25개'}")
print("-" * 78)
for p in [0.45, 0.5, 0.55, 0.6, 0.7, 0.8]:
    row = f"{p:<16}{p:<12.4f}"
    for k in [5, 11, 25]:
        row += f"{majority_accuracy(p, k):<12.4f}"
    print(row)
print("-" * 78)
print()
print("[핵심 조건]")
print("  p > 0.5 : 모델을 늘릴수록 좋아진다")
print("  p = 0.5 : 아무리 늘려도 그대로")
print("  p < 0.5 : 오히려 나빠진다")
print()
print("[더 중요한 조건 — 독립성]")
print()
print("  위 계산은 각 모델이 **서로 독립적으로** 틀린다고 가정한다.")
print("  똑같은 모델 100개를 모으면 똑같이 틀리므로 아무 소용이 없다.")
print()
print("  → 앙상블의 핵심은 **서로 다른 모델을 만드는 것**이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 모델 수에 따른 향상 ---
ax = axes[0]
ks = [1, 3, 5, 11, 21, 51]
for p, color in [(0.45, "#DC2626"), (0.5, "#94A3B8"),
                 (0.6, "#EA580C"), (0.7, "#0D9488"), (0.8, "#1E40AF")]:
    accs = [majority_accuracy(p, k) if k > 1 else p for k in ks]
    ax.plot(ks, accs, marker="o", markersize=4, linewidth=2,
            color=color, label=f"p={p}")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("모델 개수")
ax.set_ylabel("다수결 정확도")
ax.set_title("모델을 늘리면")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 오른쪽: 상관관계의 영향 (모의) ---
ax = axes[1]
p = 0.7
k = 21
# 상관이 있으면 실효 모델 수가 줄어드는 것으로 근사
correlations = np.linspace(0, 0.9, 10)
effective_k = [max(1, k * (1 - c)) for c in correlations]
accs = [majority_accuracy(p, max(1, int(ek))) for ek in effective_k]

ax.plot(correlations, accs, marker="o", linewidth=2.5, color="#EA580C")
ax.axhline(p, color="#DC2626", linestyle="--", linewidth=1.5)
ax.text(0.6, p - 0.02, "모델 1개 수준", fontsize=8, color="#DC2626")
ax.set_xlabel("모델 간 상관 (높을수록 비슷하게 틀림)")
ax.set_ylabel("다수결 정확도")
ax.set_title(f"모델이 서로 비슷하면 효과가 준다 (p={p}, k={k})")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("오른쪽 그래프의 요지 (개념적 근사)")
print("  모델들이 비슷할수록 앙상블 효과가 줄어든다.")
print()
print("  → 6절과 7절은 '어떻게 서로 다른 모델을 만들까'에 대한 두 가지 답이다.")

---

## 6. 배깅과 랜덤 포레스트 — 이론편 9.5절

**서로 다른 모델을 만드는 첫 번째 방법: 데이터를 다르게 준다.**

### 배깅 (Bagging = Bootstrap Aggregating)

원본 데이터에서 **복원 추출**로 표본을 만들어 각 트리를 학습시킨다.

```
원본 1000개 → 부트스트랩 표본 1000개 (중복 허용) → 트리 1
           → 부트스트랩 표본 1000개 (다른 조합)  → 트리 2
           ...
```

### 랜덤 포레스트

배깅에 **한 가지를 더한다** — 각 분할에서 특성도 무작위로 일부만 본다.

**왜 특성까지 제한하나?** 강한 특성 하나가 있으면 모든 트리가 그것부터 쓰게 되어
트리들이 비슷해진다. 5절에서 본 "독립성" 문제다.

In [ ]:
import numpy as np

print("=" * 70)
print("부트스트랩 — 복원 추출")
print("=" * 70)

np.random.seed(42)
n = 20
original = np.arange(n)

print(f"원본: {original}")
print()
for i in range(3):
    sample = np.random.choice(original, size=n, replace=True)
    unique = len(np.unique(sample))
    print(f"표본 {i+1}: {np.sort(sample)}")
    print(f"        고유 {unique}개 ({unique/n*100:.0f}%), "
          f"빠진 것 {n-unique}개")
print()

# 이론값 확인
print("이론적으로 빠지는 비율")
for size in [10, 100, 1000, 10000]:
    excluded = (1 - 1/size) ** size
    print(f"  n={size:>6}: {excluded:.4f}")
print()
print(f"n이 커지면 1/e = {np.exp(-1):.4f} 에 수렴한다.")
print("  즉 각 표본에서 약 37% 는 빠지고 63% 만 쓰인다.")
print()
print("[OOB (Out-Of-Bag) 평가]")
print("  빠진 37% 로 검증할 수 있다 — 별도 검증 세트가 필요 없다.")
print("  랜덤 포레스트의 oob_score=True 옵션이 이것이다.")

In [ ]:
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("=" * 78)
print("단일 트리 vs 배깅 vs 랜덤 포레스트")
print("=" * 78)

models = {
    "단일 트리 (제한 없음)": DecisionTreeClassifier(random_state=42),
    "단일 트리 (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "배깅 (트리 100개)": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100, random_state=42),
    "랜덤 포레스트 (100개)": RandomForestClassifier(
        n_estimators=100, random_state=42),
}

print(f"{'모델':<28}{'학습':<12}{'시험':<12}{'격차'}")
print("-" * 78)

results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    tr = accuracy_score(y_tr, model.predict(X_tr))
    te = accuracy_score(y_te, model.predict(X_te))
    results[name] = {"train": tr, "test": te, "model": model}
    print(f"{name:<28}{tr:<12.4f}{te:<12.4f}{tr-te:+.4f}")

print("-" * 78)
print()

single = results["단일 트리 (제한 없음)"]["test"]
forest = results["랜덤 포레스트 (100개)"]["test"]
print(f"단일 트리 → 랜덤 포레스트: {single:.4f} → {forest:.4f} ({forest-single:+.4f})")
print()
print("[주목할 점]")
print("  앙상블도 학습 정확도는 1.0 에 가깝다 — 각 트리가 제한 없이 자라므로")
print("  하지만 **시험 정확도가 훨씬 높다** — 트리들의 오류가 상쇄되기 때문")
print()
print("  즉 앙상블은 '외우는 것'을 막는 게 아니라")
print("  '각자 다르게 외운 것을 평균내어' 일반화를 얻는다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 70)
print("트리 개수에 따른 성능")
print("=" * 70)

n_trees = [1, 3, 5, 10, 25, 50, 100, 200]
rf_scores, oob_scores = [], []

import warnings

for n in n_trees:
    # 트리가 적으면 OOB 추정이 불안정하다는 경고가 나온다 — 정상 동작
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        rf = RandomForestClassifier(n_estimators=n, random_state=42,
                                    oob_score=(n >= 25), n_jobs=-1)
    rf.fit(X_tr, y_tr)
    rf_scores.append(accuracy_score(y_te, rf.predict(X_te)))
    oob_scores.append(rf.oob_score_ if n >= 25 else np.nan)

print(f"{'트리 수':<12}{'시험 정확도':<16}{'OOB 점수'}")
print("-" * 70)
for n, s, o in zip(n_trees, rf_scores, oob_scores):
    oob_str = f"{o:.4f}" if not np.isnan(o) else "—"
    print(f"{n:<12}{s:<16.4f}{oob_str}")
print("-" * 70)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(n_trees, rf_scores, marker="o", linewidth=2.5,
        color="#0D9488", label="시험 정확도")
valid = [(n, o) for n, o in zip(n_trees, oob_scores) if not np.isnan(o)]
if valid:
    ns, os_ = zip(*valid)
    ax.plot(ns, os_, marker="s", linewidth=2, linestyle="--",
            color="#EA580C", label="OOB 점수")
ax.set_xscale("log")
ax.set_xlabel("트리 개수 (로그)")
ax.set_ylabel("정확도")
ax.set_title("트리를 늘리면")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print()
print("초반에 급격히 오르다가 평평해진다 (수확 체감).")
print("  보통 100~500개면 충분하며, 더 늘려도 크게 나아지지 않는다.")
print()
print("[중요] 트리를 늘려도 과대적합하지 않는다")
print("  각 트리는 독립적으로 학습되므로, 개수는 '평균의 안정성'만 높인다.")
print("  이것이 부스팅(7절)과 결정적으로 다른 점이다.")

---

## 7. 부스팅 — 이론편 9.5절

**서로 다른 모델을 만드는 두 번째 방법: 앞 모델의 실수를 다음 모델이 고친다.**

| | 배깅 | 부스팅 |
|---|---|---|
| 학습 방식 | **병렬** (독립적) | **순차** (앞의 결과를 봄) |
| 각 모델 | 깊은 트리 | **얕은 트리** (약한 학습기) |
| 목표 | 분산 감소 | **편향 감소** |
| 과대적합 | 개수 늘려도 안전 | **개수가 많으면 위험** |

**그래디언트 부스팅**은 앞 모델의 **잔차(오차)**를 다음 모델이 학습한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

print("=" * 70)
print("부스팅의 원리 — 잔차를 학습한다")
print("=" * 70)

# 회귀로 원리를 보는 것이 직관적이다
np.random.seed(0)
X_reg = np.linspace(0, 10, 100).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + np.random.randn(100) * 0.2

# 순차적으로 잔차를 학습
predictions = np.zeros(len(y_reg))
stages = []
learning_rate = 0.5

for stage in range(4):
    residual = y_reg - predictions          # 아직 못 맞힌 부분
    tree = DecisionTreeRegressor(max_depth=2).fit(X_reg, residual)
    predictions = predictions + learning_rate * tree.predict(X_reg)
    mse = ((y_reg - predictions) ** 2).mean()
    stages.append((stage + 1, predictions.copy(), residual.copy(), mse))
    print(f"  {stage+1}단계: MSE {mse:.4f}")

print()
print("단계를 거칠수록 오차가 준다.")

fig, axes = plt.subplots(2, 4, figsize=(15, 6))

for i, (n, pred, resid, mse) in enumerate(stages):
    # 위: 예측
    ax = axes[0, i]
    ax.scatter(X_reg, y_reg, s=12, alpha=0.5, color="#94A3B8", label="데이터")
    ax.plot(X_reg, pred, linewidth=2.5, color="#1E40AF", label="누적 예측")
    ax.set_title(f"{n}단계 (MSE {mse:.3f})", fontsize=10)
    if i == 0:
        ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 아래: 잔차
    ax = axes[1, i]
    ax.scatter(X_reg, resid, s=12, alpha=0.6, color="#DC2626")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{n}단계 시작 시 잔차", fontsize=9)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("아래 줄을 보면 잔차가 점점 0 근처로 모인다.")
print("  각 단계의 트리는 '아직 못 맞힌 부분'만 학습한다.")

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 78)
print("부스팅 vs 랜덤 포레스트")
print("=" * 78)

models_boost = {
    "랜덤 포레스트 (100)": RandomForestClassifier(n_estimators=100, random_state=42),
    # ── GradientBoostingClassifier 파라미터 ──────────────────────
    #   n_estimators    부스팅 단계 수.  기본값 100
    #                   **많으면 과대적합** (랜덤 포레스트와 다른 점)
    #   learning_rate   각 트리의 기여도.  기본값 0.1
    #                   낮을수록 안전. 예: 0.01~0.3
    #                   n_estimators 와 반비례로 맞춘다
    #                   (lr 낮추면 n_estimators 를 늘린다)
    #   max_depth       각 트리 깊이.  기본값 3
    #                   부스팅은 얕은 트리를 쓴다. 예: 3~8
    #   subsample       각 단계에서 쓸 표본 비율.  기본값 1.0
    #                   0.8 정도로 낮추면 확률적 부스팅(과대적합 완화)
    #   validation_fraction  조기 종료용 검증 비율.  기본값 0.1
    #   n_iter_no_change     조기 종료 기준.  기본값 None
    #                        예: 10 — 10회 개선 없으면 중단
    # ──────────────────────────────────────────────────────────────
    "그래디언트 부스팅 (100)": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "부스팅 lr=0.05": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05, random_state=42),
    "부스팅 lr=0.5": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.5, random_state=42),
}

print(f"{'모델':<28}{'학습':<12}{'시험':<12}{'격차'}")
print("-" * 78)
for name, model in models_boost.items():
    model.fit(X_tr, y_tr)
    tr = accuracy_score(y_tr, model.predict(X_tr))
    te = accuracy_score(y_te, model.predict(X_te))
    print(f"{name:<28}{tr:<12.4f}{te:<12.4f}{tr-te:+.4f}")
print("-" * 78)
print()
print("[학습률의 역할]")
print("  각 단계의 기여를 얼마나 반영할지 정한다.")
print("  낮으면 천천히 안전하게, 높으면 빠르지만 과대적합 위험")
print()
print("  10장에서 다룰 경사하강법의 학습률과 같은 개념이다.")
print("  실무에서는 낮은 학습률 + 많은 트리 조합을 많이 쓴다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 70)
print("트리 개수에 따른 차이 — 부스팅은 과대적합할 수 있다")
print("=" * 70)

n_range = [10, 25, 50, 100, 200, 400]
rf_te, gb_te, gb_tr = [], [], []

for n in n_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    rf_te.append(accuracy_score(y_te, rf.predict(X_te)))

    gb = GradientBoostingClassifier(n_estimators=n, learning_rate=0.3,
                                     max_depth=5, random_state=42)
    gb.fit(X_tr, y_tr)
    gb_tr.append(accuracy_score(y_tr, gb.predict(X_tr)))
    gb_te.append(accuracy_score(y_te, gb.predict(X_te)))

print(f"{'개수':<10}{'RF 시험':<14}{'GB 학습':<14}{'GB 시험'}")
print("-" * 70)
for n, r, gt, ge in zip(n_range, rf_te, gb_tr, gb_te):
    print(f"{n:<10}{r:<14.4f}{gt:<14.4f}{ge:.4f}")
print("-" * 70)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(n_range, rf_te, marker="o", linewidth=2.5,
        color="#0D9488", label="랜덤 포레스트 (시험)")
ax.plot(n_range, gb_te, marker="s", linewidth=2.5,
        color="#EA580C", label="부스팅 (시험)")
ax.plot(n_range, gb_tr, marker="^", linewidth=2, linestyle="--",
        color="#DC2626", label="부스팅 (학습)")
ax.set_xlabel("트리 개수")
ax.set_ylabel("정확도")
ax.set_title("부스팅은 개수를 늘리면 과대적합할 수 있다")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("부스팅의 학습 정확도가 1.0 으로 붙는 것을 보라.")
print("  앞 단계의 오차를 계속 줄이다 보면 노이즈까지 맞추게 된다.")
print()
print("[대책] 조기 종료")
print("  검증 성능이 나빠지기 시작하면 멈춘다.")
print("  sklearn 은 n_iter_no_change, validation_fraction 옵션 제공")

---

## 8. 특성 중요도 — 이론편 9.5절

트리 계열의 실용적 장점 하나. **어느 특성이 중요했는지 알 수 있다.**

각 특성이 분할에 쓰이며 줄인 불순도를 합산해 계산한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

feature_names = [f"특성{i}" for i in range(X.shape[1])]

print("=" * 78)
print("특성 중요도")
print("=" * 78)

# 1) 불순도 기반 (기본)
imp_gini = rf.feature_importances_
order = np.argsort(imp_gini)[::-1]

print("[불순도 감소 기반]")
print(f"{'순위':<8}{'특성':<12}{'중요도':<14}{'막대'}")
print("-" * 78)
for rank, idx in enumerate(order[:8], 1):
    bar = "█" * int(imp_gini[idx] * 200)
    print(f"{rank:<8}{feature_names[idx]:<12}{imp_gini[idx]:<14.4f}{bar}")
print("-" * 78)
print(f"합계: {imp_gini.sum():.4f}  (항상 1이 되도록 정규화된다)")
print()

# 2) 순열 중요도
perm = permutation_importance(rf, X_te, y_te, n_repeats=10,
                              random_state=42, n_jobs=-1)
imp_perm = perm.importances_mean
order_perm = np.argsort(imp_perm)[::-1]

print("[순열 중요도 — 값을 섞었을 때 성능이 얼마나 떨어지나]")
print(f"{'순위':<8}{'특성':<12}{'중요도':<14}{'표준편차'}")
print("-" * 78)
for rank, idx in enumerate(order_perm[:8], 1):
    print(f"{rank:<8}{feature_names[idx]:<12}{imp_perm[idx]:<14.4f}"
          f"{perm.importances_std[idx]:.4f}")
print("-" * 78)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

top_k = 8

# --- 왼쪽: 불순도 기반 ---
ax = axes[0]
idxs = order[:top_k][::-1]
ax.barh(range(top_k), imp_gini[idxs], color="#1E40AF")
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_names[i] for i in idxs], fontsize=9)
ax.set_xlabel("중요도")
ax.set_title("불순도 감소 기반")
ax.grid(axis="x", alpha=0.3)

# --- 오른쪽: 순열 중요도 ---
ax = axes[1]
idxs_p = order_perm[:top_k][::-1]
ax.barh(range(top_k), imp_perm[idxs_p],
        xerr=perm.importances_std[idxs_p], capsize=3, color="#0D9488")
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_names[i] for i in idxs_p], fontsize=9)
ax.set_xlabel("중요도 (성능 하락폭)")
ax.set_title("순열 중요도")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# 두 방법의 순위 비교
top_gini = set(order[:5])
top_perm = set(order_perm[:5])
overlap = len(top_gini & top_perm)

print("=" * 78)
print("두 방법 비교")
print("=" * 78)
print(f"  상위 5개 중 겹치는 것: {overlap}개")
print()
print(f"{'방법':<20}{'장점':<26}{'주의점'}")
print("-" * 78)
print(f"{'불순도 기반':<20}{'계산이 빠름 (학습 중 산출)':<26}{'범주 많은 특성 과대평가'}")
print(f"{'순열 중요도':<20}{'실제 성능 영향 반영':<26}{'느림, 상관 특성에 취약'}")
print("-" * 78)
print()
print("[주의] 중요도 ≠ 인과관계")
print("  '중요하다'는 것은 예측에 유용했다는 뜻이지")
print("  그 특성이 결과를 **일으킨다**는 뜻이 아니다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **9.4** | **지니 [10,0]=0, [5,5]=0.5, [8,2]=0.32** | **일치** ✓ |
| 9.4 | 정보 이득 0.5 / 0.18 / 0.0 | 일치 ✓ |
| 9.4 | 분할점 탐색 | 직접 구현 ✓ |
| 8.5 | 트리의 과대적합 | 깊이별 측정 ✓ |
| 9.5 | 앙상블의 다수결 효과 | 이항분포 계산 ✓ |
| 9.5 | 부트스트랩에서 37% 제외 | 검증 ✓ |

### 세 가지 모델 정리

| 모델 | 만드는 법 | 강점 | 약점 |
|---|---|---|---|
| 결정트리 | 불순도가 가장 줄어드는 분할 반복 | 해석 쉬움, 스케일 무관 | **과대적합** |
| 랜덤 포레스트 | 부트스트랩 + 특성 무작위 | 안정적, 개수 늘려도 안전 | 해석 어려움 |
| 그래디언트 부스팅 | 잔차를 순차 학습 | 성능 높음 | 과대적합 위험, 느림 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 지니 vs 엔트로피 | 거의 같음, 지니가 빠름 |
| 정보 이득 | 자식을 **크기로 가중평균** |
| 트리 과대적합 | 제한 없으면 학습 100% |
| 앙상블 조건 | $p > 0.5$ **그리고 서로 달라야** |
| 부트스트랩 | 약 37%가 빠짐 → OOB 평가 |
| RF 트리 수 | 늘려도 안전 (독립 학습) |
| 부스팅 트리 수 | **많으면 과대적합** (순차 학습) |
| 특성 중요도 | 인과관계가 아님 |

### 언제 무엇을 쓸까

| 상황 | 권장 |
|---|---|
| 규칙을 설명해야 함 | 결정트리 (깊이 제한) |
| 무난한 기본 선택 | 랜덤 포레스트 |
| 성능이 최우선 | 그래디언트 부스팅 (튜닝 필요) |
| 특성이 아주 많음 | 랜덤 포레스트 |
| 데이터가 아주 큼 | LightGBM·XGBoost 계열 |

### 다음 장

**10. 퍼셉트론과 MLP — 직접 만들어 보는 신경망** — 트리와는 또 다른 접근이다.
8장에서 봤던 XOR 문제를 이번에는 **신경망으로** 푼다.